## **Pre-Processing**

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(),"..", "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import build_dataset as bd
import importlib
importlib.reload(bd)

from build_dataset import DATA_DIR, FILE_ORDER, TARGET, LABEL_MAP

pd.set_option("display.max_colwidth", None)

**1. Schema Verification**

The CIC-IDS2017 dataset captures network traffic data over a week (July 3-7 2017). Data is split into 8 csv files based on day of week and attack type, but was captured continuously with the same methodology, making merging valid.

Build out functions used in this notebook can be found in `../src/build_dataset.py `.

In [3]:
# VERIFY SCHEMAS OF INDIVIDUAL DATA FILES
schemas = bd.read_schemas()
print(f"{bd.n_distinct_schemas(schemas)} distinct schema(s) across data files.")

1 distinct schema(s) across data files.


**2. Within file cleaning then merge separate files**

`merge_files` cleans the data for each day before appending to a single parquet file. 
- Strips whitepace and drops duplicate columns
- Repairs mojibake dash in `Web Attack` labels
- Drops rows with nulls with null values in numeric features and within-file duplicates (REVISIT VALIDITY)
- Downcasts numerics to float 32

In [4]:
# CLEAN AND MERGE INDIVIDUAL DATA FILES
merged_path, per_file_counts = bd.merge_files()
print("\nMerged to path:", merged_path)

Monday-WorkingHours.pcap_ISCX.csv: 502,650 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 421,626 rows
Wednesday-workingHours.pcap_ISCX.csv: 610,492 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 164,179 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 252,790 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 184,044 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 213,777 rows
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 223,082 rows

Merged to path: c:\Users\lixin\portfolio\cybersecurity_project\data\merged_rawlabels.parquet


**3. Cross-file cleaning, constant column removal, and column renaming**

`clean_merged` passes for cross-file duplicates and constant columns, then converts column names to camel case. 

In [5]:
# FILTER MERGED DATA
stats = bd.clean_merged()
print("Constant columns:", stats["constant_cols"])
print("\nWritten to path:", stats["path"])

Number of inter-file duplicates: 74,660
Number of constant columns: 8
Rows written: 2,497,980
Constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Written to path: c:\Users\lixin\portfolio\cybersecurity_project\data\merge_complete.parquet


In [6]:
# CONFIRM FINAL COLUMN NAMES
print(stats["columns"])

['DestinationPort', 'FlowDuration', 'TotalFwdPackets', 'TotalBackwardPackets', 'TotalLengthOfFwdPackets', 'TotalLengthOfBwdPackets', 'FwdPacketLengthMax', 'FwdPacketLengthMin', 'FwdPacketLengthMean', 'FwdPacketLengthStd', 'BwdPacketLengthMax', 'BwdPacketLengthMin', 'BwdPacketLengthMean', 'BwdPacketLengthStd', 'FlowBytesS', 'FlowPacketsS', 'FlowIatMean', 'FlowIatStd', 'FlowIatMax', 'FlowIatMin', 'FwdIatTotal', 'FwdIatMean', 'FwdIatStd', 'FwdIatMax', 'FwdIatMin', 'BwdIatTotal', 'BwdIatMean', 'BwdIatStd', 'BwdIatMax', 'BwdIatMin', 'FwdPshFlags', 'FwdUrgFlags', 'FwdHeaderLength', 'BwdHeaderLength', 'FwdPacketsS', 'BwdPacketsS', 'MinPacketLength', 'MaxPacketLength', 'PacketLengthMean', 'PacketLengthStd', 'PacketLengthVariance', 'FinFlagCount', 'SynFlagCount', 'RstFlagCount', 'PshFlagCount', 'AckFlagCount', 'UrgFlagCount', 'CweFlagCount', 'EceFlagCount', 'DownUpRatio', 'AveragePacketSize', 'AvgFwdSegmentSize', 'AvgBwdSegmentSize', 'SubflowFwdPackets', 'SubflowFwdBytes', 'SubflowBwdPackets', 